# 04 – Feature Engineering

Cyclical day-of-year encoding from `date_of_record`.

- Adds `doy_sin`, `doy_cos`
- Drops redundant `month` / `season` text columns
- Adds `station_id` (name + lat/lon/elevation) to disambiguate colliding station names
- Keeps `station_name`, `state`, `district` as metadata

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path("..").resolve()
if not (BASE / "data" / "processed" / "clean_dataset.csv").exists():
    BASE = Path(".").resolve()

CLEAN_PATH = BASE / "data" / "processed" / "clean_dataset.csv"
OUT_PATH = BASE / "data" / "processed" / "feature_engineered.csv"
OUT_V2 = BASE / "data" / "processed" / "feature_engineered_v2.csv"

df = pd.read_csv(CLEAN_PATH)
df["date_of_record"] = pd.to_datetime(df["date_of_record"])
print("Loaded:", df.shape)
df.head()

In [ ]:
n_rows_before = len(df)
assert df["date_of_record"].isna().sum() == 0, "Found unparseable dates"
print(f"Rows: {n_rows_before:,}")
print("Columns:", df.columns.tolist())

In [ ]:
day_of_year = df["date_of_record"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * day_of_year / 366)
df["doy_cos"] = np.cos(2 * np.pi * day_of_year / 366)
df = df.drop(columns=["month", "season"], errors="ignore")

In [ ]:
# Disambiguate stations that share a name but differ in location/elevation
df["station_id"] = (
    df["station_name"].astype(str)
    + "_"
    + df["latitude"].round(2).astype(str)
    + "_"
    + df["longitude"].round(2).astype(str)
    + "_"
    + df["elevation"].astype(int).astype(str)
)
n_dup = int(df.groupby(["station_id", "date_of_record"]).size().gt(1).sum())
print(f"station_name unique: {df['station_name'].nunique()}")
print(f"station_id unique:   {df['station_id'].nunique()}")
print(f"duplicate (station_id, date) groups: {n_dup}")
assert n_dup == 0

In [ ]:
assert len(df) == n_rows_before
assert df["doy_sin"].between(-1, 1).all()
assert df["doy_cos"].between(-1, 1).all()
assert df.isna().sum().sum() == 0

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.drop(columns=["station_id"]).to_csv(OUT_PATH, index=False)
df.to_csv(OUT_V2, index=False)
print(f"Saved {OUT_PATH} (without station_id)")
print(f"Saved {OUT_V2} (with station_id) shape={df.shape}")
df[["date_of_record", "doy_sin", "doy_cos", "station_id"]].head()